# Dữ liệu Animal Condition
**Nguồn trong đề:** Lab 2 — bài tập 2.2.4 (SVM)

Notebook độc lập cho một tập dữ liệu; code lấy từ các ô tương ứng của notebook Lab2 (giữ nguyên, ô nào chỉnh có ghi `# [Chỉnh]`).

In [1]:
# Thư viện dùng chung (đề import rải rác ở nhiều ô; gom lại để notebook chạy độc lập)
import os
if os.path.basename(os.getcwd()) == 'notebooks':   # chạy notebook từ thư mục notebooks/ thì lùi ra thư mục gốc chứa data/
    os.chdir('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import matplotlib as mpl
import graphviz
from sklearn import tree, svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
from sklearn.preprocessing import OneHotEncoder
animal = pd.read_csv('data/Animal-Condition-Predict.csv')
print(animal.shape)
print(animal.isnull().sum())
print(animal['Dangerous'].value_counts())
animal = animal.dropna(subset=['Dangerous'])      # bỏ 2 dòng thiếu nhãn
Xa = animal.drop(columns='Dangerous'); ya = animal['Dangerous']
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.2, random_state=42, stratify=ya)
# Tất cả đặc trưng là biến phân loại nên mã hóa one-hot trước khi đưa vào SVM
pipe_a = make_pipeline(OneHotEncoder(handle_unknown='ignore'), svm.SVC())
param_a = {'svc__kernel':['linear','rbf'], 'svc__C':[0.1,1,10,100]}
gs_a = GridSearchCV(pipe_a, param_a, cv=5, scoring='accuracy')
gs_a.fit(Xa_tr, ya_tr)
print("Tham số tốt nhất:", gs_a.best_params_, " Accuracy CV: %.4f" % gs_a.best_score_)
pred = gs_a.predict(Xa_te)
print("Accuracy test: %.4f" % accuracy_score(ya_te, pred))
print(confusion_matrix(ya_te, pred))
print(classification_report(ya_te, pred, zero_division=0))

(871, 7)
AnimalName    0
symptoms1     0
symptoms2     0
symptoms3     0
symptoms4     0
symptoms5     0
Dangerous     2
dtype: int64
Dangerous
Yes    849
No      20
Name: count, dtype: int64


Tham số tốt nhất: {'svc__C': 1, 'svc__kernel': 'linear'}  Accuracy CV: 0.9871
Accuracy test: 1.0000
[[  4   0]
 [  0 170]]
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         4
         Yes       1.00      1.00      1.00       170

    accuracy                           1.00       174
   macro avg       1.00      1.00      1.00       174
weighted avg       1.00      1.00      1.00       174

